# LSTM Autoencoder Anomaly Detection
This notebook runs the end-to-end pipeline using the modular Python code.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path("..").resolve()
sys.path.append(str(ROOT))

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

In [ ]:
from src.models.lstm_autoencoder import LSTMAutoencoder, load_config
from src.data.lstm_autoencoder_preprocessing import load_raw_data, clean_raw_data, split_train_val, fit_scaler, scale_features, create_sequences, save_processed_data, save_scaler
from src.features.lstm_autoencoder_features import engineer_features

In [ ]:
from src.training.lstm_autoencoder_training import prepare_dataloaders, train_lstm_autoencoder, resolve_device, save_model, plot_training_history
from src.evaluation.lstm_autoencoder_evaluation import reconstruction_errors, compute_threshold, flag_anomalies, align_scores_with_timestamps, plot_reconstruction_errors, plot_feature_with_anomalies

In [ ]:
config = load_config(ROOT / "src" / "config" / "lstm_autoencoder_config.yaml")

In [ ]:
raw_df = load_raw_data(ROOT / config["data"]["raw_path"], config["data"]["timestamp_col"])
clean_df = clean_raw_data(raw_df, config["data"]["timestamp_col"])

In [ ]:
feat_df, feature_cols = engineer_features(clean_df)
processed_path = save_processed_data(feat_df, ROOT / config["data"]["processed_dir"], config["data"]["processed_filename"])

In [ ]:
train_df, val_df = split_train_val(feat_df, config["data"]["val_split"])
scaler = fit_scaler(train_df, feature_cols)

In [ ]:
train_values = scale_features(train_df, feature_cols, scaler)
val_values = scale_features(val_df, feature_cols, scaler)
train_seq = create_sequences(train_values, config["data"]["window_size"], config["data"]["stride"])

In [ ]:
val_seq = create_sequences(val_values, config["data"]["window_size"], config["data"]["stride"])
train_loader, val_loader = prepare_dataloaders(train_seq, val_seq, config["training"]["batch_size"])

In [ ]:
device = resolve_device(config["training"]["device"])
model = LSTMAutoencoder.from_config(ROOT / "src" / "config" / "lstm_autoencoder_config.yaml").to(device)
history = train_lstm_autoencoder(model, train_loader, val_loader, config["training"], device)

In [ ]:
save_model(model, ROOT / config["training"]["model_path"])
save_scaler(scaler, ROOT / config["training"]["scaler_path"])

In [ ]:
plot_training_history(history)

In [ ]:
all_values = scale_features(feat_df, feature_cols, scaler)
all_seq = create_sequences(all_values, config["data"]["window_size"], config["data"]["stride"])
errors = reconstruction_errors(model, all_seq, config["training"]["batch_size"], device)

In [ ]:
threshold = compute_threshold(errors, config["training"]["threshold_quantile"])
score_df = align_scores_with_timestamps(feat_df[config["data"]["timestamp_col"]], errors, config["data"]["window_size"], config["data"]["stride"])

In [ ]:
score_df["is_anomaly"] = flag_anomalies(score_df["reconstruction_error"].values, threshold)
anomaly_times = score_df.loc[score_df["is_anomaly"], "timestamp"]

In [ ]:
plot_reconstruction_errors(score_df, threshold)

In [ ]:
plot_feature_with_anomalies(feat_df, "particle_count", anomaly_times)